# 🗳️ ElectioAnalytics — Modèle Binaire
**Algorithme :** Régression Logistique  
**Cible :** `cible_binaire` — DROITE (0) vs EXCEPTION (1)  
**Comparaison :** Version A (avec historique électoral) vs Version B (socio-éco pur)

---

## ⚙️ 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    RocCurveDisplay
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

FINAL_PATH  = '../data/final'
MODELS_PATH = '../data/models'
os.makedirs(MODELS_PATH, exist_ok=True)

# Colonnes cibles à exclure de la détection automatique des features
COLS_CIBLES = ['cible_binaire', 'cible_multiclasse', 'cible_transition_enc']

print('✅ Imports OK')

---
## 📂 1 — Chargement des datasets

In [ ]:
datasets = {}
for version in ['A', 'B']:
    train = pd.read_csv(f'{FINAL_PATH}/ml_train_2017_v{version}.csv', dtype={'code_geo': str})
    test  = pd.read_csv(f'{FINAL_PATH}/ml_test_2022_v{version}.csv',  dtype={'code_geo': str})
    features = [c for c in train.columns
                if (c.endswith('_scaled') or c.endswith('_enc'))
                and c not in COLS_CIBLES]
    datasets[version] = {'train': train, 'test': test, 'features': features}
    print(f'Version {version} — {len(features)} features : {features}')

print(f'\nTrain : {len(datasets["A"]["train"])} communes (2017)')
print(f'Test  : {len(datasets["A"]["test"])} communes (2022)')

In [ ]:
# Distribution de la cible
for version in ['A', 'B']:
    y_train = datasets[version]['train']['cible_binaire'].values
    y_test  = datasets[version]['test']['cible_binaire'].values
    print(f'Version {version} — Train : DROITE={(y_train==0).sum()} / EXCEPTION={(y_train==1).sum()} '
          f'| Test : DROITE={(y_test==0).sum()} / EXCEPTION={(y_test==1).sum()}')

---
## 🤖 2 — Cross-Validation (k=5 stratifié)

In [ ]:
resultats = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for version in ['A', 'B']:
    train    = datasets[version]['train']
    features = datasets[version]['features']
    X_train  = train[features].values
    y_train  = train['cible_binaire'].values

    model = LogisticRegression(
        class_weight='balanced',
        C=0.1,
        max_iter=1000,
        random_state=42,
        solver='lbfgs'
    )

    cv_results = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring=['accuracy', 'f1', 'roc_auc'],
        return_train_score=True
    )

    resultats[version] = {
        'model': model, 'features': features,
        'cv': cv_results,
        'X_train': X_train, 'y_train': y_train
    }
    print(f'\n── Version {version} ──')
    df_cv = pd.DataFrame({
        'Métrique':  ['Accuracy', 'F1 Score', 'AUC-ROC'],
        'Train moy': [cv_results['train_accuracy'].mean(),
                      cv_results['train_f1'].mean(),
                      cv_results['train_roc_auc'].mean()],
        'Val moy':   [cv_results['test_accuracy'].mean(),
                      cv_results['test_f1'].mean(),
                      cv_results['test_roc_auc'].mean()],
        'Val std':   [cv_results['test_accuracy'].std(),
                      cv_results['test_f1'].std(),
                      cv_results['test_roc_auc'].std()],
    }).set_index('Métrique').round(3)
    df_cv['Overfit?'] = (df_cv['Train moy'] - df_cv['Val moy']).apply(
        lambda x: '⚠️ Oui' if x > 0.1 else '✅ Non'
    )
    display(df_cv)

---
## 🧪 3 — Évaluation sur le test 2022

In [ ]:
for version in ['A', 'B']:
    train    = datasets[version]['train']
    test     = datasets[version]['test']
    features = datasets[version]['features']

    X_train = train[features].values
    y_train = train['cible_binaire'].values
    X_test  = test[features].values
    y_test  = test['cible_binaire'].values

    model = resultats[version]['model']
    model.fit(X_train, y_train)

    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    resultats[version].update({
        'y_test': y_test, 'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'acc':  accuracy_score(y_test, y_pred),
        'f1':   f1_score(y_test, y_pred),
        'auc':  roc_auc_score(y_test, y_pred_proba),
        'cm':   confusion_matrix(y_test, y_pred),
        'coefs': pd.DataFrame({
            'feature': features,
            'coefficient': model.coef_[0]
        }).sort_values('coefficient', ascending=False),
        'test_df': test.copy(),
        'X_test': X_test
    })

    joblib.dump(model, f'{MODELS_PATH}/model_binaire_v{version}.pkl')

    print(f'\n── Version {version} ──')
    print(classification_report(
        y_test, y_pred,
        target_names=['DROITE', 'EXCEPTION'],
        digits=3
    ))

---
## 📊 4 — Comparaison Version A vs Version B

In [ ]:
df_comp = pd.DataFrame({
    'Version A': [
        f"{resultats['A']['acc']:.3f}",
        f"{resultats['A']['f1']:.3f}",
        f"{resultats['A']['auc']:.3f}",
        f"{resultats['A']['cv']['test_accuracy'].mean():.3f} ± {resultats['A']['cv']['test_accuracy'].std():.3f}"
    ],
    'Version B': [
        f"{resultats['B']['acc']:.3f}",
        f"{resultats['B']['f1']:.3f}",
        f"{resultats['B']['auc']:.3f}",
        f"{resultats['B']['cv']['test_accuracy'].mean():.3f} ± {resultats['B']['cv']['test_accuracy'].std():.3f}"
    ],
    'Écart A-B': [
        f"{resultats['A']['acc'] - resultats['B']['acc']:+.3f}",
        f"{resultats['A']['f1']  - resultats['B']['f1']:+.3f}",
        f"{resultats['A']['auc'] - resultats['B']['auc']:+.3f}",
        '—'
    ]
}, index=['Accuracy (test)', 'F1 Score (test)', 'AUC-ROC (test)', 'CV Accuracy (train)'])

print('📊 Tableau comparatif A vs B :')
display(df_comp)

ecart = resultats['A']['acc'] - resultats['B']['acc']
if ecart > 0.15:
    msg = '→ Le passé électoral est indispensable pour prédire'
elif ecart > 0.08:
    msg = '→ Le passé électoral apporte un signal significatif'
else:
    msg = '→ Les conditions socio-éco suffisent à prédire'
print(f'\n   Écart A-B : {ecart:+.3f}  {msg}')

---
## 🟦 5 — Matrices de Confusion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, version in zip(axes, ['A', 'B']):
    cm     = resultats[version]['cm']
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    # Annotations : nombre + pourcentage
    annot = np.array([
        [f"{cm[i,j]}\n({cm_pct[i,j]:.1f}%)" for j in range(2)]
        for i in range(2)
    ])

    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
                xticklabels=['DROITE', 'EXCEPTION'],
                yticklabels=['DROITE', 'EXCEPTION'],
                linewidths=1.5, cbar=False,
                annot_kws={'size': 12})

    acc = resultats[version]['acc']
    f1  = resultats[version]['f1']
    desc = 'Avec historique électoral' if version == 'A' else 'Socio-éco pur (prédiction 2027)'
    ax.set_title(f'Version {version} — {desc}\nAccuracy={acc:.1%}  F1={f1:.3f}',
                 fontweight='bold')
    ax.set_xlabel('Prédit', fontsize=11)
    ax.set_ylabel('Réel',   fontsize=11)

plt.suptitle('Matrices de Confusion — Test 2022', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/confusion_binaire.png', dpi=120)
plt.show()

---
## 📈 6 — Courbes ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = {'A': '#e74c3c', 'B': '#3498db'}
labels = {'A': 'Version A — Avec historique', 'B': 'Version B — Socio-éco pur'}

for version in ['A', 'B']:
    RocCurveDisplay.from_predictions(
        resultats[version]['y_test'],
        resultats[version]['y_pred_proba'],
        name=f"{labels[version]} (AUC={resultats[version]['auc']:.3f})",
        color=colors[version],
        ax=ax
    )

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Modèle aléatoire (AUC=0.500)')
ax.set_title('Courbes ROC — Modèle Binaire\nVersion A vs Version B',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlabel('Taux de faux positifs', fontsize=11)
ax.set_ylabel('Taux de vrais positifs', fontsize=11)
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/roc_binaire.png', dpi=120)
plt.show()

---
## 🔢 7 — Coefficients du modèle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, version in zip(axes, ['A', 'B']):
    coefs  = resultats[version]['coefs']
    colors = ['#e74c3c' if c > 0 else '#3498db' for c in coefs['coefficient']]
    ax.barh(coefs['feature'], coefs['coefficient'], color=colors, edgecolor='white')
    ax.axvline(x=0, color='black', linewidth=1)
    desc = 'Avec historique' if version == 'A' else 'Socio-éco pur'
    ax.set_title(f'Coefficients — Version {version}\n{desc}', fontweight='bold')
    ax.set_xlabel('Impact sur P(EXCEPTION)\n(rouge = ↑ probabilité exception / bleu = ↓)')

plt.suptitle('Coefficients de la Régression Logistique', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/coefficients_binaire.png', dpi=120)
plt.show()

print('\n📋 Interprétation des coefficients Version B (prédiction 2027) :')
for _, row in resultats['B']['coefs'].iterrows():
    sens = 'augmente' if row['coefficient'] > 0 else 'diminue'
    print(f"   {row['feature']:<35} : {row['coefficient']:+.3f} → {sens} P(EXCEPTION)")

---
## 🏙️ 8 — Probabilités de bascule par commune (Version B)

In [ ]:
test_b = resultats['B']['test_df'].copy()
test_b['proba_exception'] = resultats['B']['y_pred_proba']
test_b['y_pred']          = resultats['B']['y_pred']
test_b['y_reel']          = resultats['B']['y_test']
test_b['correct']         = (test_b['y_pred'] == test_b['y_reel'])

# Top 30 communes avec la plus haute probabilité d'exception
top30 = test_b.sort_values('proba_exception', ascending=False).head(30)

couleurs = {'DROITE': '#c0392b', 'CENTRE': '#f39c12', 'GAUCHE': '#2980b9', 'DIVERS': '#7f8c8d'}
colors   = [couleurs.get(b, '#95a5a6') for b in top30['bloc_vainqueur']]

fig, ax = plt.subplots(figsize=(16, 6))
bars = ax.bar(range(len(top30)), top30['proba_exception'],
              color=colors, alpha=0.85, edgecolor='white')

# Marquer les erreurs de prédiction
for i, (_, row) in enumerate(top30.iterrows()):
    if not row['correct']:
        ax.text(i, row['proba_exception'] + 0.01, '✗',
                ha='center', fontsize=10, color='black')

ax.axhline(y=0.5, color='black', linestyle='--', alpha=0.5, label='Seuil décision = 0.5')
ax.set_xticks(range(len(top30)))
ax.set_xticklabels(top30['libelle_commune'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('P(EXCEPTION)', fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title(
    'Top 30 communes — Probabilité d\'exception (Version B, test 2022)\n'
    '🔴 DROITE  🟠 CENTRE  🔵 GAUCHE  (couleur = résultat réel)  ✗ = erreur de prédiction',
    fontweight='bold'
)
ax.legend()
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/probas_communes_binaire.png', dpi=120)
plt.show()

---
## 🔍 9 — Analyse des erreurs (Version B)

In [ ]:
faux_negatifs = test_b[(test_b['y_reel'] == 1) & (test_b['y_pred'] == 0)]
faux_positifs = test_b[(test_b['y_reel'] == 0) & (test_b['y_pred'] == 1)]

print(f'❌ Faux négatifs : {len(faux_negatifs)} communes')
print('   → Communes qui ont échappé à DROITE mais non détectées\n')
display(faux_negatifs[['libelle_commune', 'bloc_vainqueur', 'proba_exception']]
        .sort_values('proba_exception', ascending=False)
        .reset_index(drop=True))

print(f'\n⚠️  Faux positifs : {len(faux_positifs)} communes')
print('   → Communes prédites EXCEPTION mais restées DROITE\n')
display(faux_positifs[['libelle_commune', 'bloc_vainqueur', 'proba_exception']]
        .sort_values('proba_exception', ascending=False)
        .head(10)
        .reset_index(drop=True))

---
## 📋 10 — Bilan Final

In [ ]:
print('=' * 60)
print('📋 BILAN MODÈLE BINAIRE — Régression Logistique')
print('=' * 60)

for version in ['A', 'B']:
    r      = resultats[version]
    cv_acc = r['cv']['test_accuracy'].mean()
    cv_std = r['cv']['test_accuracy'].std()
    desc   = 'Avec historique électoral' if version == 'A' else 'Socio-éco pur (2027)'
    print(f"""
Version {version} — {desc}
   CV Accuracy (k=5)  : {cv_acc:.3f} ± {cv_std:.3f}
   Test Accuracy 2022 : {r['acc']:.3f}  ({r['acc']*100:.1f}%)
   F1 Score           : {r['f1']:.3f}
   AUC-ROC            : {r['auc']:.3f}""")

ecart = resultats['A']['acc'] - resultats['B']['acc']
print(f"""
Écart A-B
   Accuracy : {ecart:+.3f}
   AUC-ROC  : {resultats['A']['auc'] - resultats['B']['auc']:+.3f}

Fichiers sauvegardés
   data/models/model_binaire_vA.pkl
   data/models/model_binaire_vB.pkl
   data/models/confusion_binaire.png
   data/models/roc_binaire.png
   data/models/coefficients_binaire.png
   data/models/probas_communes_binaire.png

🚀 Prochaine étape : model_multiclasse.ipynb""")